# ⚛️ NQS (Neural Quantum States) — Google Colab 计算工作台（免 Google Drive 版）

### 三步上手（适配 Colab Compute Units + 后台运行）：
1. **选择 GPU（及后台运行）**：点击顶部菜单 **`Runtime` -> `Change runtime type`**，入门算例选 **`T4 GPU`**（最省 Units，约 1~2 units/小时），大算例可选 **`L4` / `A100`**；如页面有 **Background execution（后台执行）** 开关可一并打开，这样本地电脑断电也不会中断计算！
2. **配置 W&B 密钥（免挂载 Google Drive）**：点击左侧边栏 **🔑 Secrets**，添加名称为 `WANDB_API_KEY` 的密钥（在 https://wandb.ai/authorize 获取）并开启读取权限。所有能量曲线和断点 Checkpoint 都会自动同步到 W&B 云端，**完全不需要挂载 Google Drive**。
3. **一键运行 + 算完自动省电**：点击 **`Runtime` -> `Run all`**。若无人值守运行，可勾选第 4 格的 `AUTO_DISCONNECT_WHEN_DONE`，算完保存结果后会**立刻自动释放 GPU 实例**，绝不白烧 Compute Units！

In [ ]:
# @title 1. 检查 GPU 硬件并安装 JAX / NetKet 环境
import shutil
import subprocess
import sys

has_gpu = shutil.which("nvidia-smi") is not None
if has_gpu:
    print("✅ 检测到 NVIDIA GPU！正在安装 CUDA 加速版 NetKet...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "netket[cuda]", "flax", "optax", "wandb", "pyyaml", "matplotlib"])
else:
    print("⚠️ 当前未开启 GPU（建议在 Runtime -> Change runtime type 中切换为 T4 GPU），正在安装 CPU 版 NetKet...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "netket", "flax", "optax", "wandb", "pyyaml", "matplotlib"])

import jax
print("JAX 加速设备列表:", jax.devices())

In [ ]:
# @title 2. 拉取最新项目代码（已预填你的 GitHub 账号 jiahao116）
import os
import subprocess
import sys

REPO_URL = "https://github.com/jiahao116/NQS.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
WORK_DIR = "/content/NQS"

if not os.path.exists(WORK_DIR):
    subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, WORK_DIR])
else:
    subprocess.check_call(["git", "-C", WORK_DIR, "pull", "origin", BRANCH])

os.chdir(WORK_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print(f"✅ 代码已就绪，当前工作目录: {os.getcwd()}")

In [ ]:
# @title 3. 连接 W&B 云端看板与断点仓库（默认无需 Google Drive！）
import os

USE_WANDB = True            # @param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = False  # @param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/NQS_checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"✅ Checkpoint 保存至 Google Drive: {CHECKPOINT_DIR}")
else:
    CHECKPOINT_DIR = "/content/NQS/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print("✅ 免 Google Drive 模式：Checkpoint 将自动同步备份至 W&B Artifacts 云端！")

if USE_WANDB:
    try:
        from google.colab import userdata
        api_key = userdata.get("WANDB_API_KEY")
        if api_key:
            os.environ["WANDB_API_KEY"] = api_key
            print("✅ 已从 Colab Secrets 自动读取 WANDB_API_KEY！")
    except Exception:
        pass
    if not os.environ.get("WANDB_API_KEY"):
        import wandb
        wandb.login()

In [ ]:
# @title 4. 启动 / 自动续算 NQS 实验（支持算完自动断开省 Units）
import time
from IPython.display import Image, display
from nqs.config import load_config
from nqs.train import run_experiment

# 可选实验配置：
# - configs/tfim_1d_gpt.yaml       (推荐首跑：1D 横场伊辛临界点 + Mini-GPT 波函数)
# - configs/tfim_1d_rbm.yaml       (经典对照：1D 横场伊辛 + RBM + 量子自然梯度 SR)
# - configs/heisenberg_1d_gpt.yaml (进阶实验：1D 反铁磁海森堡链 + 复相位 GPT 波函数)
CONFIG_PATH = "configs/tfim_1d_gpt.yaml"  # @param ["configs/tfim_1d_gpt.yaml", "configs/tfim_1d_rbm.yaml", "configs/heisenberg_1d_gpt.yaml"]
FRESH_START = False                 # @param {type:"boolean"}
AUTO_DISCONNECT_WHEN_DONE = False   # @param {type:"boolean"}

cfg = load_config(CONFIG_PATH)
summary = run_experiment(
    cfg=cfg,
    checkpoint_dir_override=CHECKPOINT_DIR,
    no_wandb=not USE_WANDB,
    fresh_start=FRESH_START,
)

plot_path = summary.get("plot_path")
if plot_path:
    display(Image(filename=plot_path))

if AUTO_DISCONNECT_WHEN_DONE:
    print("⏳ 结果与 Checkpoint 已全部上传 W&B，5 秒后自动释放 Colab GPU 实例以节省 Compute Units...")
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()